In [281]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sparse


import importlib
import sys
sys.path.append('/Users/giovanniconcheri/Desktop/ResearchJob/Student-Research-Job/Chebyshev')
sys.path.append('/Users/giovanniconcheri/Desktop/ResearchJob/Student-Research-Job/TCI')
sys.path.append('/Users/giovanniconcheri/Desktop/ResearchJob/Student-Research-Job/Correlation_function')
sys.path.append("/home/t30/all/go56vod/Desktop/Student-Research-Job/Chebyshev")
sys.path.append("/home/t30/all/go56vod/Desktop/Student-Research-Job/TCI")
sys.path.append("/home/t30/all/go56vod/Desktop/Student-Research-Job/Correlation_function")


import ED_Cs_Lsites as ED
import TCI_singlesite as TCI_single
import TCI_Lsite_accumulative_v1 as TCI_multi
import tensor_cross_interpolation as tci
import MPS 
import Chebyshev as Cheby
import peak_to_MPS as peakMPS
from convolution_as_MPO import construct_convolution_MPO
import convolution_as_MPO as mpo


importlib.reload(ED)
importlib.reload(TCI_single)
importlib.reload(TCI_multi)
importlib.reload(MPS)
importlib.reload(Cheby)
importlib.reload(peakMPS)
importlib.reload(mpo)

<module 'convolution_as_MPO' from '/Users/giovanniconcheri/Desktop/ResearchJob/Student-Research-Job/Noisy/convolution_as_MPO.py'>

## TCI algorithm on noisy data

In [282]:
# define Hamiltonian terms
L = 11
n = 11
dt = 1e-2
models = ['J = 1, g = 2 , Ising Model Ham. = H','J =1 g = 2 Ham. = H - k*Hzz', 'J = 1 g = 0.5, Ham. = H - h * Hxx', 'J = 1 g = 0.5, Ham. = H - k*Hzz - h*Hxx']
threshold = 30
N = 2**n
D = L
g_par = 0.1


In [283]:
# Generate Hamiltonian
Hlist = [ED.gen_Ham(L = L, model = 0), ED.gen_Ham(L = L, model = 2)]

In [284]:
Cs_theo = ED.correlator(H= Hlist[0], L = L, dt = dt,  n = n)
print(Cs_theo.shape)

for i in range(Cs_theo.shape[0]):
    for k in range(Cs_theo.shape[1]):
        if Cs_theo.real[i,k] > 1:
            print('i,k: (', i, ',', k, '), ', Cs_theo[i,k])
            Cs_theo[i,k] = 1. + 1j* Cs_theo.imag[i,k]
        if Cs_theo.imag[i,k] > 1:
            print('i,k: (', i, ',', k, '), ', Cs_theo[i,k])
            Cs_theo[i,k] = Cs_theo.real[i,k] + 1j

Expectation value $\bra{\psi_0} X_{L/2} \ket{psi_0}$=  4.1729792341192077e-16
(11, 2048)
i,k: ( 5 , 0 ),  (1.0000000000000016+0j)


In [285]:
N_shots = 2**13 #8192 rough estimation by Bernhard
# clip to avoid tiny negative values from rounding
std_real = np.sqrt(np.clip(1 - Cs_theo.real**2, 0, None)) / np.sqrt(N_shots)
std_imag = np.sqrt(np.clip(1 - Cs_theo.imag**2, 0, None)) / np.sqrt(N_shots)

np.random.seed(0)  # optional
Cs_noise_real = np.random.normal(loc=0.0, scale=std_real)
Cs_noise_imag = np.random.normal(loc=0.0, scale=std_imag)

print(Cs_noise_real[0])

Cs_noise = Cs_theo + Cs_noise_real + 1j * Cs_noise_imag
print("shapes:", Cs_theo.shape, Cs_noise.shape)

[ 0.01948883  0.00442084  0.01081286 ...  0.00108669 -0.00514087
  0.01373149]
shapes: (11, 2048) (11, 2048)


In [286]:
diff_noisevstheo = Cs_theo-Cs_noise #should be difference between 2 matrices
err_max_noisevstheo = np.max(np.abs(diff_noisevstheo))/np.max(np.abs(Cs_theo))
print("Max error (noise vs theo): ", err_max_noisevstheo)
err_2_noisevstheo = np.linalg.norm(diff_noisevstheo)/np.linalg.norm(Cs_theo)
print("2-norm error (noise vs theo): ", err_2_noisevstheo)

Max error (noise vs theo):  0.05144266433826994
2-norm error (noise vs theo):  0.046755419202998506


In [287]:
# D = L #already defined above

func_vals_theo = Cs_theo.T #Cs is in the form (X,T)
func_vals_noise = Cs_noise.T

#first we define the function f(t)
# which returns the slice of func_vals along x for a specific time 
f_t_theo = lambda *t: Cs_theo.reshape((D,) + (2,) * n)[:,*t]
f_t_noise = lambda *t: Cs_noise.reshape((D,) + (2,) * n)[:,*t]

In [288]:
func_noise = TCI_multi.function(f_t_noise)
chi = 15

As, _, eval, err_2, err_max, func_interp_noise = TCI_multi.accumulative_tensor_cross_interpolation(func_noise,         # function to be interpolated
                                   func_vals_noise,    
                                   D,
                                   L=n,          # number of MPS tensors
                                   iters=chi-1)      # number of back-and-forth sweeps


err_max:  0.05624551701835813
err_2:  0.05400100027426711

repeated evaluations:  43750
unique evaluations 1410
unique + repeated:  45160
total evaluations:  45160



In [289]:
func_noise.cache

{(0,
  0,
  np.int32(0),
  np.int32(0),
  np.int32(0),
  np.int32(0),
  np.int32(0),
  np.int32(0),
  np.int32(0),
  np.int32(0),
  np.int32(0)): array([0.03138499-0.00899577j, 0.00107573-0.00327601j,
        0.0599556 -0.00718025j, 0.0994426 -0.01328591j,
        0.25316137+0.00199687j, 1.        -0.01457152j,
        0.25762969-0.00318938j, 0.09855339+0.00422966j,
        0.0444511 -0.00020328j, 0.01688546+0.03194271j,
        0.0142879 -0.01996479j]),
 (0,
  1,
  np.int32(0),
  np.int32(0),
  np.int32(0),
  np.int32(0),
  np.int32(0),
  np.int32(0),
  np.int32(0),
  np.int32(0),
  np.int32(0)): array([-0.30830803+0.19209582j, -0.07383833+0.34438997j,
        -0.42274696+0.11552227j, -0.14740196+0.01779316j,
         0.16778218-0.08645434j,  0.35112859+0.25616322j,
         0.15733075-0.09616977j, -0.15264106+0.02011962j,
        -0.42208587+0.10867456j, -0.07192746+0.32981734j,
        -0.30504488+0.19044498j]),
 (1,
  0,
  np.int32(0),
  np.int32(0),
  np.int32(0),
  np.int32(0),
 

In [290]:
for A in As:
    print(A.shape) 

(11, 1, 2, 15)
(15, 2, 15)
(15, 2, 15)
(15, 2, 15)
(15, 2, 15)
(15, 2, 15)
(15, 2, 15)
(15, 2, 8)
(8, 2, 4)
(4, 2, 2)
(2, 2, 1)


## Least squares optimization

In [291]:
fw_shift = mpo.binary_addition_MPO(L, l=+1, modular=False)
print("fw_shift shapes:", [w.shape for w in fw_shift])
id_ = [np.eye(2)[None, None] for _ in range(L)]
id_[0] *= -1.
Ws = mpo.add_MPO(fw_shift, id_)

fw_shift shapes: [(1, 2, 2, 2), (2, 2, 2, 2), (2, 2, 2, 2), (2, 2, 2, 2), (2, 2, 2, 2), (2, 2, 2, 2), (2, 2, 2, 2), (2, 2, 2, 2), (2, 2, 2, 2), (2, 2, 2, 2), (2, 1, 2, 2)]


In [293]:
for w in Ws:
    print(w.shape)

(1, 3, 2, 2)
(3, 3, 2, 2)
(3, 3, 2, 2)
(3, 3, 2, 2)
(3, 3, 2, 2)
(3, 3, 2, 2)
(3, 3, 2, 2)
(3, 3, 2, 2)
(3, 3, 2, 2)
(3, 3, 2, 2)
(3, 1, 2, 2)


1. Fare SVD del TCI per ridurre bond dimension
2. fare dmrg-like optimization

Devo fare un codice simile a quello d_dmrg.py. 

IDEA: 
- trovare tutti gli R (termini a destra)
- poi partendo da site 0, updateare L e R fino a site L-1 updateando M ottimizzando i termini 1 e 2
- e poi fare il backward sweep da site L-1 a site 0.

Capire come fare la parte dell'ottimizzazione del termine 2.

In [294]:
import least_squares as lls
importlib.reload(lls)
from copy import deepcopy
As_mod = [deepcopy(A) for A in As]

In [295]:
ls_eng = lls.Engine(As_mod, Ws, func_noise.cache)

In [296]:
# ls_eng.sweep(lr=1e-5, lambda_=2)
ls_eng.sweep(method='als', lambda_=2, batch_size=200) # batch_size is the number of samples to use for stochastic optimization
As_new = ls_eng.As# Scegli l'indice spaziale/temporale da visualizzare nel plot
idx_to_plot = 1

xs = np.arange(N)

# Plot comparison
fig, axs = plt.subplots(ncols=2, figsize=(7,3), dpi=300)
axs[0].plot(np.arange(N), func_vals_theo[:,0], '-', label = 'theory')
axs[0].plot(np.arange(N), func_interp_noise[:,0], ':', label = 'noisy full TCI')
axs[0].plot(np.arange(N), MPS.interpolate_func(As_new)[:,0], '--', label = 'noisy ls TCI')
axs[0].set(xlabel=r'Input $x$',
           ylabel=r'Convolution $(f \ast \mathcal{N})(x)$')
# axs[0].set_xlim([900,1100])
# axs[0].set_ylim([-0.2, 0.2])
axs[0].legend()

axs[1].plot(xs, func_vals_theo[:,0] - MPS.interpolate_func(As_new)[:,0], '-')
axs[1].set(xlabel=r'Input $x$',
           ylabel='Diff. of exact and MPS values')

plt.tight_layout()
plt.show()

err_max_ls, err_2_ls = MPS.errors(As_new, func_vals_noise)
print("Max error w.r.t. noisy function: ", err_max, err_max_ls)
print("2-norm error w.r.t. noisy function: ", err_2, err_2_ls)

AttributeError: 'Engine' object has no attribute 'N_total'